In [ ]:
from mxdataspark import MXData
from mxdataspark.core import MXDataClient


In [ ]:
from importlib.metadata import version
print(f'MXData Spark package version: {version("mxdataspark")}')


In [ ]:
# DBTITLE 1,Widget Parameters
dbutils.widgets.text("model_name", "")
dbutils.widgets.text("entity_type", "ALL")
dbutils.widgets.text("load_group", "")
dbutils.widgets.text("run_id", "")
dbutils.widgets.text("catalog_name", "")
dbutils.widgets.text("job_concurrency", "")


In [ ]:
model_name = dbutils.widgets.get("model_name")
entity_type = dbutils.widgets.get("entity_type")
load_group = dbutils.widgets.get("load_group")
run_id = dbutils.widgets.get("run_id")
job_concurrency = dbutils.widgets.get("job_concurrency")
job_concurrency = int(job_concurrency) if int(job_concurrency or 0) > 0 else None

catalog_name = dbutils.widgets.get("catalog_name").lower()
spark.catalog.setCurrentCatalog(catalog_name)


In [ ]:
client = MXDataClient()
model_entities = client.get_model_group_entities(model_name, catalog_name, entity_type)

print("MXData.queue_model Configs:")
print(f"{model_entities=}")
print(f"{job_concurrency=}")
print(f"{load_group=}")
print(f"{run_id=}")


In [ ]:
# DBTITLE 1,Main
metrics = (
    MXData.model("batch")
    .queue_model(model_entities)
    .process_batch(
        workers=job_concurrency, load_group=load_group, run_id=run_id, logging=False
    )
    .metrics("dict")
)


In [ ]:
FAIL = False

for metric in metrics:
    if not metric["success"]:
        FAIL = True
    client.set_model_status(load_group, run_id, metric)


In [ ]:
if FAIL:
    print(f"A model has failed! Check the Metrics.")
    for metric in metrics:
        if not metric["success"]:
            print(metric)
    raise ValueError("The model build failed! Check ObjectModel. Exiting... ")
